# Plantilla de modelo – JarvisTEC

**Integrante(s):** _completar_  
**Dataset:** _nombre y enlace_

Esta plantilla sigue las secciones de la rúbrica (total: 5 pts por modelo). Instrucciones:

1. Duplicar este archivo como `notebooks/NN_nombre.ipynb`.
2. Completar la celda de **Configuración**.
3. Ejecutar de arriba hacia abajo. Donde diga **✍️ COMPLETAR**, escribir texto propio: la rúbrica califica el análisis y las conclusiones, no solo el código.
4. Borrar este bloque de instrucciones al terminar.

| Sección | Puntos |
|---|---|
| 1. Análisis del problema | 0.5 |
| 2. Entendimiento de los datos | 0.5 |
| 3. Exploración de los datos | 0.5 |
| 4. Modelo | 2 |
| 5. Evaluación | 1 |
| 6. Conclusión | 0.5 |

## Configuración

In [ ]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

warnings.filterwarnings("ignore")

# ------------------ EDITAR ESTAS VARIABLES ------------------
NOMBRE_MODELO   = "vino"                       # debe coincidir con `name` en backend/models/vino.py
RUTA_DATOS      = "../data/winequality.csv"    # ruta al CSV (la carpeta data/ no se sube a git)
COLUMNA_OBJETIVO = "quality"                   # variable a predecir
TIPO_TAREA      = "clasificacion"              # "clasificacion" o "regresion"
ES_SERIE_TEMPORAL = False                      # True para bitcoin, aguacate, etc. (split cronológico)
COLUMNA_FECHA   = None                         # nombre de la columna de fecha si es serie temporal
COLUMNAS_EXCLUIR = []                          # ids u otras columnas que no deben usarse como entrada
SEMILLA         = 42
# -------------------------------------------------------------

ARTIFACTS = Path("../artifacts")
ARTIFACTS.mkdir(exist_ok=True)
np.random.seed(SEMILLA)
assert TIPO_TAREA in ("clasificacion", "regresion")

## 1. Análisis del problema

**✍️ COMPLETAR** (3–6 líneas):

- ¿Qué se quiere predecir y por qué es útil? ¿Quién usaría este resultado?
- ¿Es clasificación, regresión, serie temporal o recomendación? ¿Por qué?
- ¿Qué métrica define el éxito y por qué es la adecuada para este problema?
- ¿Qué artículo científico o referencia respalda el enfoque? (agregar a la bibliografía)

## 2. Entendimiento de los datos

In [ ]:
df = pd.read_csv(RUTA_DATOS)
print("Filas, columnas:", df.shape)
display(df.head())

In [ ]:
resumen = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "nulos": df.isna().sum(),
    "% nulos": (df.isna().mean() * 100).round(2),
    "únicos": df.nunique(),
})
display(resumen)
print("Filas duplicadas:", df.duplicated().sum())
display(df.describe(include="all").T)

**✍️ COMPLETAR:** describir cada variable relevante (significado, unidades, tipo), el origen de los datos, el tamaño, los valores faltantes y cualquier problema de calidad detectado (duplicados, valores imposibles, columnas inútiles).

## 3. Exploración de los datos

In [ ]:
y_col = COLUMNA_OBJETIVO
fig, ax = plt.subplots(figsize=(6, 3.5))
if TIPO_TAREA == "clasificacion":
    df[y_col].value_counts().sort_index().plot(kind="bar", ax=ax)
    ax.set_title(f"Distribución de clases: {y_col}")
    display((df[y_col].value_counts(normalize=True) * 100).round(2).rename("% de cada clase"))
else:
    df[y_col].plot(kind="hist", bins=40, ax=ax)
    ax.set_title(f"Distribución de {y_col}")
plt.tight_layout(); plt.show()

In [ ]:
num_cols = [c for c in df.select_dtypes("number").columns if c not in COLUMNAS_EXCLUIR + [y_col]]
df[num_cols].hist(bins=30, figsize=(12, 3 * ((len(num_cols) + 3) // 4)), layout=(-1, 4))
plt.suptitle("Distribución de variables numéricas", y=1.02); plt.tight_layout(); plt.show()

In [ ]:
if len(num_cols) > 1:
    corr = df[num_cols + [y_col]].select_dtypes("number").corr()
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90)
    ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns)
    plt.colorbar(im); ax.set_title("Correlaciones"); plt.tight_layout(); plt.show()
    if pd.api.types.is_numeric_dtype(df[y_col]):
        display(corr[y_col].drop(y_col).sort_values(key=abs, ascending=False).rename("correlación con el objetivo"))

In [ ]:
cat_cols = [c for c in df.select_dtypes(exclude="number").columns if c not in COLUMNAS_EXCLUIR + [y_col, COLUMNA_FECHA]]
for c in cat_cols[:6]:
    print(f"\n{c}: {df[c].nunique()} categorías")
    display(df[c].value_counts().head(10))

**✍️ COMPLETAR:** ¿qué patrones, relaciones, desbalances o valores atípicos se observan? ¿Qué decisiones de preprocesamiento se toman a partir de esto (imputar, eliminar, transformar, balancear)?

## 4. Modelo

Se compara una línea base con varios modelos mediante validación cruzada y se elige el mejor. Todo el preprocesamiento va dentro de un `Pipeline`, así el backend solo recibe datos crudos y evita fugas de información.

> **Series temporales:** con `ES_SERIE_TEMPORAL = True` el split es cronológico (el test son los últimos datos) y la validación cruzada usa `TimeSeriesSplit`. Nunca mezclar el orden temporal al partir los datos.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, KFold, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

data = df.drop_duplicates().copy()
if ES_SERIE_TEMPORAL and COLUMNA_FECHA:
    data[COLUMNA_FECHA] = pd.to_datetime(data[COLUMNA_FECHA])
    data = data.sort_values(COLUMNA_FECHA).reset_index(drop=True)

X = data.drop(columns=[COLUMNA_OBJETIVO] + COLUMNAS_EXCLUIR + ([COLUMNA_FECHA] if COLUMNA_FECHA else []))
y = data[COLUMNA_OBJETIVO]
num_features = X.select_dtypes("number").columns.tolist()
cat_features = X.select_dtypes(exclude="number").columns.tolist()

if ES_SERIE_TEMPORAL:
    corte = int(len(X) * 0.8)
    X_train, X_test, y_train, y_test = X.iloc[:corte], X.iloc[corte:], y.iloc[:corte], y.iloc[corte:]
    cv = TimeSeriesSplit(n_splits=5)
else:
    strat = y if TIPO_TAREA == "clasificacion" and y.value_counts().min() >= 2 else None
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEMILLA, stratify=strat)
    cv = StratifiedKFold(5, shuffle=True, random_state=SEMILLA) if strat is not None else KFold(5, shuffle=True, random_state=SEMILLA)

preprocesador = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_features),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_features),
])
print(f"Train: {X_train.shape}  Test: {X_test.shape}")
print("Numéricas:", num_features); print("Categóricas:", cat_features)

In [ ]:
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier, GradientBoostingRegressor

if TIPO_TAREA == "clasificacion":
    candidatos = {
        "Línea base": DummyClassifier(strategy="most_frequent"),
        "Regresión logística": LogisticRegression(max_iter=1000),
        "Random Forest": RandomForestClassifier(n_estimators=200, random_state=SEMILLA, n_jobs=-1),
        "Gradient Boosting": GradientBoostingClassifier(random_state=SEMILLA),
    }
    scoring, mayor_es_mejor = "f1_macro", True
else:
    candidatos = {
        "Línea base": DummyRegressor(strategy="mean"),
        "Ridge": Ridge(),
        "Random Forest": RandomForestRegressor(n_estimators=200, random_state=SEMILLA, n_jobs=-1),
        "Gradient Boosting": GradientBoostingRegressor(random_state=SEMILLA),
    }
    scoring, mayor_es_mejor = "neg_root_mean_squared_error", True  # RMSE negativo: más alto = mejor

filas = []
for nombre, est in candidatos.items():
    pipe = Pipeline([("prep", preprocesador), ("modelo", est)])
    s = cross_val_score(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=1)
    filas.append({"modelo": nombre, f"{scoring} (media)": s.mean(), "desv. est.": s.std()})
comparacion = pd.DataFrame(filas).sort_values(f"{scoring} (media)", ascending=False).reset_index(drop=True)
display(comparacion)

**✍️ COMPLETAR:** justificar la elección del modelo (por qué esos candidatos, por qué ganó el elegido, qué hiperparámetros importan) y citar el artículo científico que respalda el algoritmo. Opcional pero valioso: afinar hiperparámetros con `GridSearchCV` / `RandomizedSearchCV` sobre el modelo ganador.

In [ ]:
MEJOR = comparacion.loc[~comparacion["modelo"].eq("Línea base"), "modelo"].iloc[0]
print("Modelo elegido:", MEJOR)
pipeline_final = Pipeline([("prep", preprocesador), ("modelo", candidatos[MEJOR])])
pipeline_final.fit(X_train, y_train)

## 5. Evaluación

Se evalúa **una sola vez** sobre el conjunto de prueba, que el modelo nunca vio.

In [ ]:
from sklearn import metrics
pred = pipeline_final.predict(X_test)
metricas = {}

if TIPO_TAREA == "clasificacion":
    metricas = {
        "accuracy": metrics.accuracy_score(y_test, pred),
        "f1_macro": metrics.f1_score(y_test, pred, average="macro"),
        "precision_macro": metrics.precision_score(y_test, pred, average="macro", zero_division=0),
        "recall_macro": metrics.recall_score(y_test, pred, average="macro", zero_division=0),
    }
    print(metrics.classification_report(y_test, pred, zero_division=0))
    fig, ax = plt.subplots(figsize=(5, 4.5))
    metrics.ConfusionMatrixDisplay.from_predictions(y_test, pred, ax=ax, cmap="Blues", colorbar=False)
    ax.set_title("Matriz de confusión"); plt.tight_layout(); plt.show()
else:
    metricas = {
        "RMSE": metrics.mean_squared_error(y_test, pred) ** 0.5,
        "MAE": metrics.mean_absolute_error(y_test, pred),
        "R2": metrics.r2_score(y_test, pred),
    }
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].scatter(y_test, pred, alpha=0.4); lim = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
    ax[0].plot(lim, lim, "r--"); ax[0].set_xlabel("Real"); ax[0].set_ylabel("Predicho"); ax[0].set_title("Real vs. predicho")
    ax[1].hist(y_test - pred, bins=40); ax[1].set_title("Distribución de residuos")
    plt.tight_layout(); plt.show()

metricas = {k: round(float(v), 4) for k, v in metricas.items()}
display(pd.Series(metricas, name="métricas en test"))

# Comparación con la línea base sobre el mismo test (para saber si el modelo realmente aporta)
base = Pipeline([("prep", preprocesador), ("modelo", candidatos["Línea base"])]).fit(X_train, y_train)
print("Línea base en test:",
      round(metrics.f1_score(y_test, base.predict(X_test), average="macro"), 4) if TIPO_TAREA == "clasificacion"
      else round(metrics.mean_squared_error(y_test, base.predict(X_test)) ** 0.5, 4))

In [ ]:
# Importancia de variables (si el modelo la expone)
est = pipeline_final.named_steps["modelo"]
if hasattr(est, "feature_importances_"):
    nombres = pipeline_final.named_steps["prep"].get_feature_names_out()
    imp = pd.Series(est.feature_importances_, index=nombres).sort_values().tail(12)
    imp.plot(kind="barh", figsize=(6, 4), title="Importancia de variables"); plt.tight_layout(); plt.show()

**✍️ COMPLETAR:** interpretar las métricas (¿es bueno el resultado para el problema?, ¿cómo se compara con la línea base?), analizar los errores (¿qué clases o rangos falla más?) y discutir posibles causas: desbalance, ruido, pocas muestras, variables faltantes.

## Exportar para el backend

In [ ]:
joblib.dump(pipeline_final, ARTIFACTS / f"{NOMBRE_MODELO}.joblib")

# Ejemplo de entrada típica: sirve como `default_input` del modelo en backend/models/
ejemplo = X_train.median(numeric_only=True).round(3).to_dict()
for c in cat_features:
    ejemplo[c] = X_train[c].mode().iloc[0]
meta = {"modelo": MEJOR, "tarea": TIPO_TAREA, "features": X.columns.tolist(),
        "metricas_test": metricas, "default_input": ejemplo}
(ARTIFACTS / f"{NOMBRE_MODELO}.meta.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False, default=str))

print(f"Guardado en artifacts/{NOMBRE_MODELO}.joblib")
print("\nCopiar a backend/models/ como input_fields / default_input:")
print(json.dumps(ejemplo, indent=2, ensure_ascii=False, default=str))

## 6. Conclusión

**✍️ COMPLETAR** (mínimo 1 párrafo):

- Resultado principal (con la métrica) y si responde bien al problema planteado.
- Limitaciones (datos, sesgos, generalización) y riesgos de uso. En datos médicos, aclarar que **no reemplaza un diagnóstico**.
- Mejoras futuras.
- Comandos de voz asociados a este modelo: _"JarvisTEC, ..."_

### Referencias
_Agregar aquí y en la bibliografía de Overleaf (formato BibTeX)._